# A - Title Pattern Matching

In [ ]:
import sys
import os

# Add the project root directory to sys.path so that the src package can be imported 
project_root = os.path.abspath("../../")  # Adjust the relative path as needed
if project_root not in sys.path:
    sys.path.append(project_root)

### Load CRS

Download manually from OECD data explorer (searching for "CRS" -> "CRS: Creditor Reporting System (flows)") and save as `data/raw/CRS.parquet`.

In [ ]:
import pandas as pd

# Load CRS data from parquet file
crs = pd.read_parquet("../../data/raw/CRS.parquet")

Optional: Download CRS parquet file with download functions

In [ ]:
# # Import the necessary functions from src
# from src.download_crs import get_full_crs_parquet_url, download_crs_parquet

# # Get the full CRS parquet URL
# file_url = get_full_crs_parquet_url()

# # Download the CRS data
# crs = download_crs_parquet(file_url)

# # Save raw CRS to parquet
# crs.to_parquet("../../data/raw/CRS.parquet")

In [ ]:
# Reduce crs to only the columns we need
crs = crs[['project_title', 'short_description', 'long_description']]

# Only keep rows with project_title that are unique and keep the first occurrence
crs = crs.drop_duplicates(subset=['project_title'], keep='first')

# Remove row with NaN values in the project_title column (drop_dupicates keeps one None value row)
crs = crs[crs['project_title'].notna()]

### Detect Keywords 

In [ ]:
# Load text processing functions
from src.text_processing import normalize_str, detect_language, lemmatize_batch, detect_keywords, detect_acronyms, process_keywords

In [ ]:
# Load keywords from Excel file
keywords_file_path = "../../data/keywords/review/keyword_review_modernization.xlsx"
stat_keywords = pd.read_excel(keywords_file_path, sheet_name="statistics")
stat_acronyms = pd.read_excel(keywords_file_path, sheet_name="statistics acronyms")
stat_blacklist = pd.read_excel(keywords_file_path, sheet_name="stat blacklist")
gender_keywords = pd.read_excel(keywords_file_path, sheet_name="gender")
gender_acronyms = pd.read_excel(keywords_file_path, sheet_name="gender acronyms")

# Drop id column for all keywords with loop (id consistent across languages and can be used to group keywords from different languages, omitted in the following)
for df in [stat_keywords, stat_acronyms, stat_blacklist, 
           gender_keywords, gender_acronyms]:
    if 'id' in df.columns:
        df.drop(columns=['id'], inplace=True)
    
    print(f"{df.columns.tolist()}")


# Process keywords
stat_keywords = process_keywords(stat_keywords, remove_stopwords=True)
gender_keywords = process_keywords(gender_keywords, remove_stopwords=True)
stat_blacklist = process_keywords(stat_blacklist, remove_stopwords=True)

# Lowercase all strings in the acronyms dataframes
stat_acronyms = stat_acronyms.map(lambda x: x.lower() if isinstance(x, str) else x)
gender_acronyms = gender_acronyms.map(lambda x: x.lower() if isinstance(x, str) else x)

In [ ]:
# Normalize and detect title language
crs['normalized_title'] = crs['project_title'].apply(normalize_str)
crs['language'] = crs['normalized_title'].apply(detect_language)

In [ ]:
# Lemmatize in batches (for entire CRS dataset ~23min on Intel i7 Gen 11)
for lang in crs['language'].unique():
    # Filter the DataFrame for the current language
    lang_df = crs[crs['language'] == lang]

    if 'lemmatized_title' not in crs.columns:
        crs['lemmatized_title'] = None  # Initialize the column if it doesn't exist
    crs['lemmatized_title'] = crs['lemmatized_title'].astype(object)
    
    # Process the batch and update the original DataFrame
    crs.loc[lang_df.index, 'lemmatized_title'] = lemmatize_batch(lang_df['normalized_title'].tolist(), lang, batch_size=1000, remove_stopwords=True)

    # Lowercase lemmatized titles in case spacy lemmatizes to uppercase
    crs['lemmatized_title'] = crs['lemmatized_title'].str.lower()

In [ ]:
# Save after lemmatization 
crs.to_feather("../../data/processed/title_matched/crs_lemmatized_titles_wo_stopwords.feather")

# Load the lemmatized CRS data from saved checkpoint
# crs = pd.read_feather("../../data/processed//title_matched/crs_lemmatized_titles_wo_stopwords.feather")

In [ ]:
# Detect keywords in the lemmatized title
crs['stat_keywords'] = crs.apply(lambda row: detect_keywords(row['lemmatized_title'], row['language'], stat_keywords), axis=1)
crs['stat_blacklist'] = crs.apply(lambda row: detect_keywords(row['lemmatized_title'], row['language'], stat_blacklist), axis=1)
crs['gen_keywords'] = crs.apply(lambda row: detect_keywords(row['lemmatized_title'], row['language'], gender_keywords), axis=1)

# Detect acronyms in the normalized title
crs['stat_acronyms'] = crs.apply(lambda row: detect_acronyms(row['normalized_title'], row['language'], stat_acronyms), axis=1)
crs['gen_acronyms'] = crs.apply(lambda row: detect_acronyms(row['normalized_title'], row['language'], gender_acronyms), axis=1)

In [ ]:
# Save result
crs.to_feather("../../data/processed/title_matched/crs_titles_matched_wo_stopwords.feather")

In [ ]:
# Reduce crs to only rows with keywords and acronyms detected for manual inspection 
crs_reduced = crs[
    (crs['stat_keywords'].notna()) |
    (crs['stat_blacklist'].notna()) |
    (crs['gen_keywords'].notna()) |
    (crs['stat_acronyms'].notna()) |
    (crs['gen_acronyms'].notna()) 
]

# To xlsx 
crs_reduced.to_excel("../../data/processed/title_matched/crs_titles_matched_wo_stopwords.xlsx", index=False)

### Create prediction sets for text mining 

In [ ]:
# Reload raw CRS data
crs_raw = pd.read_parquet("../../data/raw/CRS.parquet")

In [ ]:
# Reduce crs_raw to necessary columns
crs_raw = crs_raw[['year', 'project_title', 'short_description', 'long_description', 'purpose_code', 'channel_code', 'donor_name', 'agency_name', 'donor_code', 'agency_code','gender', 'rmnch', 'sd_gfocus']]

In [ ]:
# Load the matched CRS data from feather file
crs_matched = pd.read_feather("../../data/processed/title_matched/crs_titles_matched_wo_stopwords.feather")

# Keep only the columns project_title, stat_keywords, stat_blacklist, gen_keywords, stat_acronyms, gen_acronyms
crs_matched = crs_matched[['project_title', 'language', 'stat_keywords', 'stat_blacklist', 'gen_keywords', 'stat_acronyms', 'gen_acronyms']]

# Rename lanaguage to title_language
crs_matched.rename(columns={'language': 'title_language'}, inplace=True)

In [ ]:
# Left join matched keywords to the the raw CRS data based on the project_title column
crs = pd.merge(crs_raw, crs_matched, on='project_title', how='left')

del crs_raw
del crs_matched

In [ ]:
# Convert chennal_code to int, gender to int, rmnch to int
crs['channel_code'] = crs['channel_code'].astype('Int64')
crs['gender'] = crs['gender'].astype('Int64')
crs['rmnch'] = crs['rmnch'].astype('Int64')
crs['agency_code'] = crs['agency_code'].astype('Int64')
crs['donor_code'] = crs['donor_code'].astype('Int64')

#### Make statistics marker & gender marker (gender marker only used for creating the training set, final gen_all marker based on more conditions)

In [ ]:
# Make statistics marker as purpose_code == 16062 | stat_keywords is not null | stat_acronyms is not null & stat_blacklist is null & pupose_code != 15250
crs['is_statistics'] = (
    (
        (crs['purpose_code'] == 16062) |    # purpose_code is 16062
        (crs['stat_keywords'].notna()) |    # stat_keywords is not null
        (crs['stat_acronyms'].notna())      # stat_acronyms is not null
    ) &
    (crs['stat_blacklist'].isna()) &        # stat_blacklist must be null
    (crs['purpose_code'] != 15250) &        # purpose_code must not be 15250
    (crs['purpose_code'] != 93010)          # purpose_code must not be 93010

)

crs['is_mining'] = (
    (crs['purpose_code'] == 15250) |        # purpose_code is 15250 
    (crs['stat_blacklist'].notna())         # stat_blacklist is not null
)

# Set is_statistics to False for title_language == it and stat_acronyms == ' ai ' 
import numpy as np
crs.loc[
    (crs['title_language'] == 'it') & 
    (crs['stat_acronyms'].apply(lambda x: isinstance(x, (list, np.ndarray)) and ' ai ' in x)), 
    'is_statistics'
] = False

crs['is_statistics'] = crs['is_statistics'].fillna(False)  # Fill NaN values with False (inlcudes those without project_title)
crs['is_mining'] = crs['is_mining'].fillna(False)  # Fill NaN values with False (inlcudes those without project_title)

In [ ]:
# Gender marker for text mining as gender == 2 that have a match in gen_keywords or gen_acronyms, but exclude those which are only detected by woman to make the gender marker for text mining as robust as possible

# Mask for only woman as keyword
only_woman = crs['gen_keywords'].apply(
    lambda x: isinstance(x, (list, np.ndarray)) and len(x) == 1 and x[0] == 'woman'
)

crs['is_gender'] = (
    (crs['gender'] == 2) &              # gender is 2
    ((crs['gen_keywords'].notna()) |    # gen_keywords is not null
    (crs['gen_acronyms'].notna())) &    # gen_acronyms is not null
    (~only_woman)
)

# Set is_gender to False if is_gender is null   
crs['is_gender'] = crs['is_gender'].fillna(False)

#### Construct sets for text mining 

In [ ]:
# Replace missing values or "" of long_descriptions with "N/A"
crs['long_description'] = crs['long_description'].replace("", "N/A")
crs['long_description'] = crs['long_description'].fillna("N/A")

# Create text mining descripton: join short_description, ": ", long_description 
crs['text_mining_description'] = crs['short_description'].astype(str) + ": " + crs['long_description'].astype(str)

In [ ]:
# Add language column to crs using detect_language function
from src.text_processing import detect_language
crs['language'] = crs['text_mining_description'].apply(detect_language)

# Set all languages that are not in ['en', 'fr', 'es', 'de', 'nl', 'pt'] to 'en'
crs['language'] = crs['language'].apply(lambda x: x if x in ['en', 'fr', 'es', 'de', 'nl', 'pt', 'it'] else 'en')

In [ ]:
# Keep only text_mining_description and gender & statistics markers
crs = crs[['text_mining_description', 'language', 'is_statistics', 'gen_keywords', 'gen_acronyms', 'is_mining', 'is_gender']]

Filter out descriptions that have conflicting values for is_statistics/is_gender (one entry True, one entry False)

In [ ]:
# Group by 'text_mining_description' and check if 'is_statistics' has more than one unique value
duplicated_stats = crs.groupby('text_mining_description')['is_statistics'].nunique()

# Filter for descriptions with more than one unique value in 'is_statistics'
conflicting_stats = duplicated_stats[duplicated_stats > 1].index

# Create a DataFrame with only the rows that match the duplicated descriptions
conflicting_descr_stat = crs[crs['text_mining_description'].isin(conflicting_stats)]['text_mining_description'].unique()

# Save conflicting_descr_stat to feather file to predict during text mining
pd.DataFrame(conflicting_descr_stat, columns=['text_mining_description']).to_feather("../../data/processed/prediction_sets/conflicting_descr_stat.feather")

In [ ]:
# Group by 'text_mining_description' and check if 'is_gender' has more than one unique value
duplicated_gen = crs.groupby('text_mining_description')['is_gender'].nunique()

# Filter for descriptions with more than one unique value in 'is_gender'
conflicting_gen = duplicated_gen[duplicated_gen > 1].index

# Create a DataFrame with only the rows that match the duplicated descriptions
conflicting_descr_gen = crs[crs['text_mining_description'].isin(conflicting_gen)]['text_mining_description'].unique()

# Save conflicting_descr_gen to feather file to predict during text mining
pd.DataFrame(conflicting_descr_gen, columns=['text_mining_description']).to_feather("../../data/processed/prediction_sets/conflicting_descr_gen.feather")

In [ ]:
# Discard rows which have conlficting is_statistics value (True & False for the same description)
stat_mining_set = crs[~crs['text_mining_description'].isin(conflicting_descr_stat)].copy()

# Drop duplicates in the stat_mining_set
stat_mining_set = stat_mining_set.drop_duplicates(subset=['text_mining_description'], keep='first')

stat_mining_set = stat_mining_set[['text_mining_description', 'language', 'is_statistics', 'is_mining']].reset_index(drop=True)

In [ ]:
# Discard rows which have conlficting is_stat value (True & False for the same description)
gen_mining_set = crs[~crs['text_mining_description'].isin(conflicting_descr_gen)].copy()

# Drop duplicates in the gen_mining_set
gen_mining_set = gen_mining_set.drop_duplicates(subset=['text_mining_description'], keep='first')

gen_mining_set = gen_mining_set[['text_mining_description', 'language', 'gen_keywords', 'gen_acronyms', 'is_gender']].reset_index(drop=True)

In [ ]:
# Save to feather
stat_mining_set.to_feather("../../data/processed/prediction_sets/stat_to_mine.feather")
gen_mining_set.to_feather("../../data/processed/prediction_sets/gen_to_mine.feather")